In [1]:
import torch
import pandas as pd
import numpy as np

print("="*70)
print("TA-GNN DATA STATISTICS VERIFICATION")
print("="*70)

# Load data
data = torch.load('data/processed/tensors/data_v4.pt', map_location='cpu', weights_only=False)

# Extract tensors
Y = data['Y'].numpy()  # Demand tensor
M = data['M'].numpy()  # Mask tensor

T, N = Y.shape
print(f"\n[1] DIMENSIONS")
print(f"    Stations (N): {N}")
print(f"    Hourly timestamps (T): {T:,}")

# Total sessions
total_sessions = int(Y.sum())
print(f"\n[2] TOTAL SESSIONS")
print(f"    Sum of demand tensor: {total_sessions:,}")

# Observed station-hours (mask = 1)
observed = int(M.sum())
print(f"\n[3] OBSERVED STATION-HOURS (mask=1)")
print(f"    Count: {observed:,}")
print(f"    Total possible (T×N): {T*N:,}")

# Positive vs zero demand
positive_mask = (Y > 0) & (M == 1)
positive_hours = int(positive_mask.sum())
zero_hours = observed - positive_hours
pct_positive = 100 * positive_hours / observed
pct_zero = 100 - pct_positive
ratio = zero_hours / positive_hours

print(f"\n[4] DEMAND DISTRIBUTION (among observed)")
print(f"    Positive station-hours (y>0): {positive_hours:,} ({pct_positive:.1f}%)")
print(f"    Zero station-hours (y=0): {zero_hours:,} ({pct_zero:.1f}%)")
print(f"    Zero-to-positive ratio: {ratio:.2f}:1")

# Positive demand statistics
pos_vals = Y[positive_mask]
print(f"\n[5] POSITIVE DEMAND VALUES")
print(f"    Mean: {pos_vals.mean():.2f}")
print(f"    Median: {np.median(pos_vals):.1f}")
print(f"    Max: {pos_vals.max():.0f}")

# Station metadata - check for coordinates
print(f"\n[6] STATION METADATA")
station_features = data['station_features'].numpy()
print(f"    Station features shape: {station_features.shape}")

# Load station metadata CSV for coordinate info
stations_df = pd.read_csv('data/processed/unified/station_metadata_v4.csv')
print(f"    Metadata columns: {list(stations_df.columns)}")

# Check for coordinate columns
if 'latitude' in stations_df.columns:
    has_coords = stations_df['latitude'].notna().sum()
    pct_coords = 100 * has_coords / len(stations_df)
    print(f"    Stations with coordinates: {has_coords} ({pct_coords:.1f}%)")
elif 'lat' in stations_df.columns:
    has_coords = stations_df['lat'].notna().sum()
    pct_coords = 100 * has_coords / len(stations_df)
    print(f"    Stations with coordinates: {has_coords} ({pct_coords:.1f}%)")
else:
    # Check station_features - often column 0,1 are lat/lon
    # Non-zero coordinates indicate valid coords
    coords = station_features[:, :2]  # Assuming first 2 cols are lat/lon
    has_coords = np.sum(np.any(coords != 0, axis=1))
    pct_coords = 100 * has_coords / N
    print(f"    Stations with non-zero coords (inferred): {has_coords} ({pct_coords:.1f}%)")

# Provider info
providers = data['provider_list']
print(f"\n[7] PROVIDERS")
print(f"    Number of providers: {len(providers)}")
print(f"    Provider names: {providers}")

# Time range
time_min = data['time_min']
print(f"\n[8] TIME RANGE")
print(f"    Start: {time_min}")
print(f"    Hours: {T:,}")
print(f"    Approx years: {T / (24*365.25):.2f}")

# ============================================
# COMPARISON TABLE
# ============================================
print("\n" + "="*70)
print("COMPARISON: PAPER vs ACTUAL")
print("="*70)
print(f"{'Metric':<35} {'Main Text':<18} {'Appendix':<18} {'ACTUAL':<18}")
print("-"*89)
print(f"{'Stations (N)':<35} {'118':<18} {'118':<18} {f'{N}':<18}")
print(f"{'Hourly timestamps (T)':<35} {'28,758':<18} {'28,758':<18} {f'{T:,}':<18}")
print(f"{'Total sessions':<35} {'83,247':<18} {'83,078':<18} {f'{total_sessions:,}':<18}")
print(f"{'Observed station-hours':<35} {'2,174,036':<18} {'1,105,292':<18} {f'{observed:,}':<18}")
print(f"{'Positive station-hours':<35} {'127,891 (5.9%)':<18} {'63,927 (5.8%)':<18} {f'{positive_hours:,} ({pct_positive:.1f}%)':<18}")
print(f"{'Zero-to-positive ratio':<35} {'~16:1':<18} {'16.29:1':<18} {f'{ratio:.2f}:1':<18}")
print(f"{'Stations with coordinates':<35} {'89 (75.4%)':<18} {'97 (82.2%)':<18} {f'{has_coords} ({pct_coords:.1f}%)':<18}")

print("\n" + "="*70)
print("WHICH VALUES TO USE IN PAPER")
print("="*70)
print("Use the ACTUAL values above. Update both Section 3 and Appendix B.")
print("="*70)

TA-GNN DATA STATISTICS VERIFICATION

[1] DIMENSIONS
    Stations (N): 118
    Hourly timestamps (T): 28,758

[2] TOTAL SESSIONS
    Sum of demand tensor: 82,950

[3] OBSERVED STATION-HOURS (mask=1)
    Count: 1,105,292
    Total possible (T×N): 3,393,444

[4] DEMAND DISTRIBUTION (among observed)
    Positive station-hours (y>0): 63,927 (5.8%)
    Zero station-hours (y=0): 1,041,365 (94.2%)
    Zero-to-positive ratio: 16.29:1

[5] POSITIVE DEMAND VALUES
    Mean: 1.30
    Median: 1.0
    Max: 16

[6] STATION METADATA
    Station features shape: (118, 3)
    Metadata columns: ['station_id', 'open_time', 'last_time', 'total_sessions', 'total_energy_kwh', 'num_evses', 'provider', 'latitude', 'longitude', 'city', 'state', 'station_name', 'operational_days', 'operational_years']
    Stations with coordinates: 97 (82.2%)

[7] PROVIDERS
    Number of providers: 5
    Provider names: ['ChargePoint', 'EVConnect', 'ElectricEra', 'Kempower', 'ZEFNET']

[8] TIME RANGE
    Start: 2022-07-31 03:00:00

In [2]:
find ~/path/to/your/project -name "*.npy" -o -name "*.pkl" -o -name "*.csv" | head -20


SyntaxError: invalid syntax (2750377964.py, line 1)